<a href="https://colab.research.google.com/github/kimcanal/robo-advisor-rl/blob/master/colab_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# RL 모듈 — Notion week-38 Colab 검증

Walk-Forward + 12지표(EW/MVO/SPY/KOSPI) + ANOVA + SHAP + 리스크태그 스텁 확인용 노트북입니다.
PR#1/#2 수정(거래비용, BIL 무위험이자율, MACD look-ahead, VecNormalize, 벤치마크, risk-tag obs)이 **포함된 `master`를 매 실행마다 새로 클론**합니다.

## Financial disclaimer (필수)

**교육·연구 목적 데모입니다. 투자 자문이 아니며, 백테스트 ≠ 미래 수익을 보장하지 않습니다.**
거래비용·슬리피지·유동성 등이 단순화되어 있을 수 있습니다. 실제 투자 결정은 본인 책임입니다.

## Notion week-38 checklist ↔ 이 노트북

| Notion 섹션 | 이 노트북에서 | 코드 포인터 |
|---|---|---|
| Architecture | 아래 파이프라인 셀 순서 | 루트 `README.md` 아키텍처 |
| Reward rationale | WF/데모가 보상 3종 학습 | `rl/README.md`, `rl/rewards.py` |
| Metrics (12) | 결과 셀: DRL vs EW/MVO/SPY/KOSPI | `rl/backtest.compute_metrics` |
| ANOVA | WF 내장 + 재표시 셀 | `rl/stats_tests.py` |
| SHAP | SHAP 셀 (Summary/Force) | `rl/shap_explain.py` / `rl.run_demo` |
| Error analysis | WF 결과·한계 기록 | `rl/README` 한계 절 |
| Risk-tag stub | 마크다운 노트 (관측 `portfolio_risk`) | `rl/risk_tags.py` |
| Disclaimer | 위 고지 | README Financial disclaimer |

## 런타임

| 모드 | 셀 | 예상 시간 | 비고 |
|---|---|---|---|
| **데모 (짧게)** | 셀 D | ~5–15분 | `run_demo` 소스텝 + 지표/ANOVA/SHAP |
| **풀 Walk-Forward** | 셀 F | ~20–40분 | ≥100k steps × 보상 3종 × ≥2 윈도우 |

- **GPU 불필요** — CPU 런타임으로 충분 (작은 MLP; GPU 이득 거의 없음).
- Colab 무료: 장시간 idle 시 끊길 수 있음 → 탭을 가끔 확인.
- **데모와 풀 WF 중 하나만** 돌리면 됩니다 (둘 다 가능하지만 시간이 겹칩니다).


## 0) Setup — 매 실행 `master` 신규 클론 + 의존성

이미 `/content/robo-advisor-rl`이 있어도 **삭제 후 재클론**해 BIL/`SPY_ohlcv`/risk-tag/벤치마크 수정이 반영되게 합니다.


In [ ]:
# --- Cell 0: fresh clone of master + deps (CPU ok; no GPU required) ---
import os
from pathlib import Path

# Colab cwd is usually /content; keep idempotent for re-runs
%cd /content
REPO = "robo-advisor-rl"
REPO_URL = "https://github.com/kimcanal/robo-advisor-rl.git"

if Path(REPO).exists():
    print(f"[setup] removing existing {REPO}/ for a fresh master clone")
    !rm -rf {REPO}

!git clone --branch master --single-branch {REPO_URL}
%cd {REPO}

!git rev-parse --short HEAD
!git log -1 --oneline

# curl_cffi: yfinance ImpersonateError 완화 (requirements에 >=0.15)
# Colab preinstalls unmaintained `gym` (NumPy 2 warning / possible native crash).
# rl/requirements.txt pins torch==2.14.0+cpu — PyPI torch is CUDA 13 and
# segfaults (exit 139) on the CPU runtime inside PPO().
!pip uninstall -y -q gym || true
!pip install -q -r rl/requirements.txt
# Colab's preinstalled triton (e.g. 3.8.0) is pulled in via torch/_dynamo
# (has_triton_package / aot_compile) and still SIGSEGVs PPO() on CPU even
# with torch==2.14.0+cpu — uninstall after install so pip cannot re-pull it.
!pip uninstall -y -q triton triton-cpu || true
print("[setup] deps installed from rl/requirements.txt (triton uninstalled)")


## 1) Data — 실제 ETF + BIL + SPY_ohlcv

라이선스상 raw CSV는 레포에 없음 → `fetch_real_data`로 받습니다.
이후 **BIL / SPY_ohlcv / risk-free 상태**를 크게 출력합니다 (조용한 0.0 폴백 금지).


In [ ]:
# --- Cell 1: fetch real data + loud BIL / SPY_ohlcv / rf status ---
!python -m rl.data.fetch_real_data

from pathlib import Path
import pandas as pd

raw = Path("rl/data/raw")
bil = raw / "BIL.csv"
spy_ohlcv = raw / "SPY_ohlcv.csv"
spy = raw / "SPY.csv"

print("\n========== DATA / RISK-FREE STATUS ==========")
print(f"BIL.csv        exists={bil.exists()}  path={bil.resolve() if bil.exists() else bil}")
print(f"SPY_ohlcv.csv  exists={spy_ohlcv.exists()}  path={spy_ohlcv.resolve() if spy_ohlcv.exists() else spy_ohlcv}")
print(f"SPY.csv        exists={spy.exists()}")

if bil.exists():
    bil_df = pd.read_csv(bil)
    print(f"[BIL] rows={len(bil_df)} cols={list(bil_df.columns)}")
    if "Date" in bil_df.columns:
        print(f"[BIL] date range: {bil_df['Date'].iloc[0]} → {bil_df['Date'].iloc[-1]}")
else:
    print("[BIL] MISSING — riskfree will try yfinance then ANNUAL_FALLBACK=0.04 (loud, not silent 0.0)")

if spy_ohlcv.exists():
    ohlcv = pd.read_csv(spy_ohlcv)
    print(f"[SPY_ohlcv] rows={len(ohlcv)} cols={list(ohlcv.columns)}")
else:
    print("[SPY_ohlcv] MISSING — Walk-Forward regime falls back to synthetic equal-weight price")

from rl.riskfree import fetch_risk_free_rate, ANNUAL_FALLBACK, BIL_CSV_PATH
rf = fetch_risk_free_rate("2019-01-01", "2026-09-29")
print(f"[riskfree] annualized rf={rf:.4f}  (BIL local preferred; fallback={ANNUAL_FALLBACK})")
print(f"[riskfree] local CSV path expected: {BIL_CSV_PATH}  exists={BIL_CSV_PATH.exists()}")
print("=============================================\n")


## 2A) Demo mode (짧게, 선택) — 파이프라인 스모크

보상 3종 학습 + 12지표(EW/MVO/SPY/KOSPI) + ANOVA + SHAP까지 **짧은 timesteps**로 확인합니다.
풀 WF를 돌릴 거면 이 셀은 **건너뛰세요**.


In [ ]:
# --- Cell 2A (OPTIONAL DEMO): short end-to-end via run_demo ---
# Skip this cell if you will run the full Walk-Forward cell below.
# ~5–15 min on Colab CPU. Produces metrics_comparison.csv + SHAP plots under rl/outputs/demo/
!python -m rl.run_demo --tickers SPY,QQQ,IWM,EFA,EEM,AGG,TLT,HYG,GLD,VNQ --timesteps 8000 --start 2019-01-01 --end 2024-12-31

from pathlib import Path
import pandas as pd

demo_metrics = Path("rl/outputs/demo/metrics_comparison.csv")
if demo_metrics.exists():
    print("=== demo 12-metric comparison (includes EW/MVO/SPY/KOSPI when available) ===")
    display(pd.read_csv(demo_metrics, index_col=0))
else:
    print(f"[demo] metrics file missing: {demo_metrics} — check cell output above")

for name in ("shap_summary_plot.png", "shap_force_plot.png"):
    p = Path("rl/outputs/demo") / name
    print(f"[demo] {name}: exists={p.exists()}  {p if p.exists() else ''}")


## 2B) Full Walk-Forward (과제 스펙) — ≥100k steps, 보상 3종, ≥2 windows

증분 CSV 저장이 코드에 내장되어 있습니다 (`_save_incremental`).
기본: `timesteps=120000`, seeds `0,1,2` (시드 늘리면 시간↑).


In [ ]:
# --- Cell 2B (FULL WF): ≥100k steps × 3 rewards × ≥2 windows ---
# Expected ~20–40 min on Colab CPU with 3 seeds. Expand seeds e.g. 0,1,2,3,4 if you have time.
# Incremental results written under rl/outputs/walk_forward/ after each window/seed.
# TORCHDYNAMO/COMPILE_DISABLE: stop torch/_dynamo from importing Colab triton (exit 139).
!TORCHDYNAMO_DISABLE=1 TORCH_COMPILE_DISABLE=1 OMP_NUM_THREADS=1 python -u -m rl.walk_forward --real --timesteps 120000 --seeds 0,1,2


## 3) 12 metrics vs EW / MVO / SPY / KOSPI

`Path.exists` 가드 — WF(또는 데모)를 먼저 실행해야 합니다.


In [ ]:
# --- Cell 3: load & display 12-metric comparison (Path.exists guards) ---
from pathlib import Path
import pandas as pd

METRIC_COLS = [
    "cumulative_return", "cagr", "ann_vol", "var_95", "cvar_95", "mdd",
    "sharpe", "sortino", "calmar", "alpha", "beta", "information_ratio",
]

wf_path = Path("rl/outputs/walk_forward/walk_forward_results.csv")
demo_path = Path("rl/outputs/demo/metrics_comparison.csv")

if wf_path.exists():
    results = pd.read_csv(wf_path)
    print(f"[metrics] loaded Walk-Forward: {wf_path}  rows={len(results)}")
    present = [c for c in METRIC_COLS if c in results.columns]
    # Strategy set should include drl_* plus equal/mvo/spy/kospi baselines
    print("strategies:", sorted(results["strategy"].dropna().unique().tolist()))
    summary = (
        results.groupby("strategy", dropna=False)[present]
        .mean(numeric_only=True)
        .sort_index()
    )
    print("=== mean 12 metrics by strategy (across windows/seeds) ===")
    display(summary)
    print("=== per-window snapshot (return / sharpe / mdd) ===")
    cols = [c for c in ["window", "strategy", "seed", "cumulative_return", "sharpe", "mdd"] if c in results.columns]
    display(results[cols])
elif demo_path.exists():
    print(f"[metrics] WF missing; showing demo metrics: {demo_path}")
    display(pd.read_csv(demo_path, index_col=0))
else:
    print(
        f"[metrics] no results yet.\n"
        f"  expected WF:   {wf_path}\n"
        f"  expected demo: {demo_path}\n"
        "Run Cell 2A (demo) and/or Cell 2B (full WF) first."
    )


## 4) ANOVA

Walk-Forward 실행 시 콘솔에 One-way(전략) + Two-way(전략×국면)가 이미 출력됩니다.
아래는 저장된 daily CSV로 다시 계산·표시합니다.


In [ ]:
# --- Cell 4: ANOVA from walk_forward_daily.csv (Path.exists guard) ---
from pathlib import Path
import pandas as pd

daily_path = Path("rl/outputs/walk_forward/walk_forward_daily.csv")

if not daily_path.exists():
    print(
        f"[anova] {daily_path} missing.\n"
        "Full WF (Cell 2B) writes this file. Demo mode uses in-memory ANOVA in run_demo logs instead."
    )
else:
    from rl.stats_tests import one_way_anova, two_way_anova

    daily = pd.read_csv(daily_path)
    print(f"[anova] daily rows={len(daily)}  regimes={daily['regime'].value_counts().to_dict() if 'regime' in daily.columns else 'n/a'}")

    # One-way: cumulative-style via mean daily return by strategy (aligned with WF verification spirit)
    groups = {
        name: g["daily_return"].dropna()
        for name, g in daily.groupby("strategy")
        if len(g["daily_return"].dropna()) > 1
    }
    if len(groups) >= 2:
        a1 = one_way_anova(groups)
        print(f"[anova] One-way (strategy daily returns): F={a1['f_stat']:.3f}, p={a1['p_value']:.4f}, eta^2={a1['eta_squared']:.4f}")
    else:
        print("[anova] need ≥2 strategies with data for one-way")

    if {"daily_return", "strategy", "regime"}.issubset(daily.columns) and daily["regime"].nunique() >= 2 and daily["strategy"].nunique() >= 2:
        a2 = two_way_anova(daily, value_col="daily_return", factor1="strategy", factor2="regime")
        print("[anova] Two-way (strategy × regime):")
        display(a2["anova_table"])
        print("eta^2:", a2["eta_squared"])
    else:
        print("[anova] skip two-way (need strategy + regime diversity)")


## 5) SHAP plots

`run_demo`가 Summary/Force plot을 `rl/outputs/demo/`에 저장합니다.
데모 셀(2A)을 이미 돌렸다면 파일을 표시만 하고, 아니면 **짧은 SHAP 전용 데모**를 돌립니다.


In [ ]:
# --- Cell 5: SHAP plots (reuse demo outputs or short run) ---
from pathlib import Path
from IPython.display import Image, display

shap_dir = Path("rl/outputs/demo")
summary_png = shap_dir / "shap_summary_plot.png"
force_png = shap_dir / "shap_force_plot.png"

if not (summary_png.exists() and force_png.exists()):
    print("[shap] plots missing — running short demo to generate SHAP (CPU ok, ~5–15 min)")
    !python -m rl.run_demo --tickers SPY,QQQ,IWM,EFA,EEM,AGG,TLT,HYG,GLD,VNQ --timesteps 5000 --start 2019-01-01 --end 2024-12-31
else:
    print("[shap] found existing demo SHAP plots")

for p in (summary_png, force_png):
    if p.exists():
        print(f"[shap] {p} ({p.stat().st_size} bytes)")
        display(Image(filename=str(p)))
    else:
        print(f"[shap] still missing: {p}")


## 6) Risk-tag stub note (Notion / README 계약)

리서치·RAG → RL 인터페이스는 `rl/risk_tags.py`입니다.

- 스키마: `{ticker, risk_score ∈ [0,1], tag, ts}`
- `PortfolioEnv` 관측 마지막 축 **`portfolio_risk`** = 보유 비중 가중 평균 `risk_score` (태그 없으면 `0.0`)
- 선택: `risk_mdd_scale`로 Safe-Guard MDD 한도를 위험도에 비례해 조임
- Colab 풀 WF는 기본적으로 태그 없이 돌아가며, 관측 차원만 계약에 맞게 확장된 상태입니다.
- 로컬 검증: `pytest rl/tests/test_risk_tags.py`

이 노트북에서는 **스텁 계약 확인용 노트**이며, RAG 실시간 연동은 팀 파이프라인 연결 후 활성화합니다.


In [ ]:
# --- Cell 6: risk-tag stub smoke (no RAG required) ---
from rl.risk_tags import mock_risk_tags, validate_risk_tags, SCHEMA_COLUMNS

print("[risk_tags] schema columns:", SCHEMA_COLUMNS)
mock = mock_risk_tags(["SPY", "QQQ"], start="2023-01-01", end="2023-06-30", n_events_per_ticker=2)
validated = validate_risk_tags(mock)
print("[risk_tags] mock events:")
display(validated.head())
print("[risk_tags] stub OK — PortfolioEnv reads these via causal merge_asof when provided")


## 7) Download results

생성된 CSV/PNG만 다운로드합니다 (`Path.exists` 가드).


In [ ]:
# --- Cell 7: download artifacts (Colab only; Path.exists guards) ---
from pathlib import Path

candidates = [
    Path("rl/outputs/walk_forward/walk_forward_results.csv"),
    Path("rl/outputs/walk_forward/walk_forward_daily.csv"),
    Path("rl/outputs/demo/metrics_comparison.csv"),
    Path("rl/outputs/demo/shap_summary_plot.png"),
    Path("rl/outputs/demo/shap_force_plot.png"),
]

existing = [p for p in candidates if p.exists()]
missing = [p for p in candidates if not p.exists()]

print("[download] present:")
for p in existing:
    print(f"  ✓ {p}")
print("[download] missing (skipped):")
for p in missing:
    print(f"  · {p}")

if not existing:
    print("[download] nothing to download — run demo and/or full WF first")
else:
    try:
        from google.colab import files
    except ImportError:
        print("[download] not on Colab (google.colab missing) — copy files from paths above")
    else:
        for p in existing:
            files.download(str(p))
